# 지역수요 프로필 분석

## 분석 목적

부산 206개 행정동의 생활인구 특성과 공공서비스 공급 현황을 결합하여
각 지역이 어떤 생활서비스 수요 특성을 가지는지 진단한다.

본 분석의 목적은 단순히 시설이 부족한 지역을 찾는 것이 아니라,
후속 단계에서 개별 유휴 공유재산의 위치 및 재산 특성과 결합하여
재산별 활용 적합유형을 판단하기 위한 지역수요 정보를 구축하는 것이다.

## 분석 축

1. 고령생활 수요
   - 60대 이상 생활인구 규모
   - 60대 이상 생활인구 구성비
   - 경로당·노인교실 공급

2. 아동생활 수요
   - 20대 미만 생활인구 규모
   - 20대 미만 생활인구 구성비
   - 지역아동센터·다함께돌봄센터 공급

3. 생활문화·학습 수요
   - 전체 생활인구 및 주거·활동 특성
   - 공공도서관 공급

4. 지역 활동 특성
   - 주거 생활인구
   - 직장 생활인구
   - 방문 생활인구

※ 각 지표를 임의의 가중치로 즉시 합산하지 않고,
분포와 관계를 먼저 확인한 뒤 활용유형 판단 기준을 설계한다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np

BASE_PATH = "/content/drive/MyDrive/부산_BigData_공모전/"

# 기존 전처리 완료 데이터 불러오기
population = pd.read_csv(
    BASE_PATH + "02_processed_data/생활인구_행정동별_202503_202512.csv",
    encoding="utf-8-sig"
)

child_supply = pd.read_csv(
    BASE_PATH + "02_processed_data/아동돌봄서비스_행정동별_공급.csv",
    encoding="utf-8-sig"
)

senior_supply = pd.read_csv(
    BASE_PATH + "02_processed_data/고령층생활서비스_행정동별_공급.csv",
    encoding="utf-8-sig"
)

print("[생활인구]")
print("shape:", population.shape)
print("columns:", population.columns.tolist())

print("\n[아동돌봄 공급]")
print("shape:", child_supply.shape)
print("columns:", child_supply.columns.tolist())

print("\n[고령층 공급]")
print("shape:", senior_supply.shape)
print("columns:", senior_supply.columns.tolist())

In [ ]:
# 행정동코드 형식 통일 및 데이터 간 코드 일치 검증
datasets = {
    "생활인구": population,
    "아동돌봄": child_supply,
    "고령층": senior_supply
}

for name, df in datasets.items():
    df["행정동코드"] = df["행정동코드"].astype(str)

    print(f"\n[{name}]")
    print("행정동 수:", len(df))
    print("코드 고유값:", df["행정동코드"].nunique())
    print("코드 중복:", df["행정동코드"].duplicated().sum())
    print("코드 결측:", df["행정동코드"].isna().sum())

population_codes = set(population["행정동코드"])
child_codes = set(child_supply["행정동코드"])
senior_codes = set(senior_supply["행정동코드"])

print("\n[생활인구 ↔ 아동돌봄]")
print("생활인구에만 존재:", len(population_codes - child_codes))
print("아동돌봄에만 존재:", len(child_codes - population_codes))

print("\n[생활인구 ↔ 고령층]")
print("생활인구에만 존재:", len(population_codes - senior_codes))
print("고령층에만 존재:", len(senior_codes - population_codes))

In [ ]:
# 연령별 생활인구 전처리 결과 불러오기
age_population = pd.read_csv(
    BASE_PATH + "02_processed_data/생활인구_행정동연령별_202503_202512.csv",
    encoding="utf-8-sig"
)

print("shape:", age_population.shape)
print("columns:", age_population.columns.tolist())

display(age_population.head())

In [ ]:
# 연령별 생활인구 구조 검증
age_population["행정동코드"] = age_population["행정동코드"].astype(str)

print("전체 행 수:", len(age_population))
print("행정동 수:", age_population["행정동코드"].nunique())

print("\n[나이대별 행 수]")
print(age_population["나이대"].value_counts().sort_index())

print("\n행정동 × 나이대 중복:",
      age_population.duplicated(["행정동코드", "나이대"]).sum())

print("\n주요 값 결측:")
print(age_population.isna().sum())

In [ ]:
# 연령별 전체활동인구 계산
age_work = age_population.copy()

age_work["전체활동인구"] = (
    age_work["평균주거인구수"]
    + age_work["평균직장인구수"]
    + age_work["평균방문인구수"]
)

# 행정동별 전체 연령 활동인구
age_total = (
    age_work.groupby("행정동코드")["전체활동인구"]
    .sum()
    .rename("연령합계_전체활동인구")
)

# 20대미만과 60대이상 추출
target_age = age_work[
    age_work["나이대"].isin(["20대미만", "60대이상"])
][["행정동코드", "나이대", "전체활동인구"]].copy()

target_age = (
    target_age
    .pivot(
        index="행정동코드",
        columns="나이대",
        values="전체활동인구"
    )
    .reset_index()
)

target_age = target_age.rename(columns={
    "20대미만": "20대미만_활동인구",
    "60대이상": "60대이상_활동인구"
})

target_age = target_age.merge(
    age_total.reset_index(),
    on="행정동코드",
    how="left"
)

# 연령 구성비
target_age["20대미만_구성비"] = (
    target_age["20대미만_활동인구"]
    / target_age["연령합계_전체활동인구"]
)

target_age["60대이상_구성비"] = (
    target_age["60대이상_활동인구"]
    / target_age["연령합계_전체활동인구"]
)

print("행정동 수:", len(target_age))
print("코드 중복:", target_age["행정동코드"].duplicated().sum())
print("결측:", target_age.isna().sum().sum())

display(target_age.head())

In [ ]:
# 기존 행정동별 전체활동인구와 연령별 합계 교차검증
activity_check = population[
    ["행정동코드", "전체활동인구"]
].merge(
    target_age[
        ["행정동코드", "연령합계_전체활동인구"]
    ],
    on="행정동코드",
    how="left"
)

activity_check["차이"] = (
    activity_check["전체활동인구"]
    - activity_check["연령합계_전체활동인구"]
)

print("최대 절대 차이:", activity_check["차이"].abs().max())
print("평균 절대 차이:", activity_check["차이"].abs().mean())

display(
    activity_check
    .assign(절대차이=lambda x: x["차이"].abs())
    .sort_values("절대차이", ascending=False)
    .head()
)

In [ ]:
# 지역수요 기본 테이블 생성
regional_profile = (
    population
    .merge(
        target_age,
        on="행정동코드",
        how="left"
    )
    .merge(
        child_supply.drop(
            columns=["행정동명", "dong_key"],
            errors="ignore"
        ),
        on="행정동코드",
        how="left"
    )
    .merge(
        senior_supply.drop(
            columns=["행정동명"],
            errors="ignore"
        ),
        on="행정동코드",
        how="left"
    )
)

print("통합 shape:", regional_profile.shape)
print("행정동 수:", len(regional_profile))
print("코드 고유값:", regional_profile["행정동코드"].nunique())
print("코드 중복:", regional_profile["행정동코드"].duplicated().sum())
print("전체 결측:", regional_profile.isna().sum().sum())

print("\n[공급량 합계 검증]")
print("지역아동센터:", regional_profile["지역아동센터"].sum())
print("다함께돌봄센터:", regional_profile["다함께돌봄센터"].sum())
print("아동돌봄시설 전체:", regional_profile["아동돌봄시설_전체"].sum())
print("경로당:", regional_profile["경로당"].sum())
print("노인교실:", regional_profile["노인교실"].sum())
print("고령층생활서비스 전체:", regional_profile["고령층생활서비스_전체"].sum())

display(regional_profile.head())

In [ ]:
# 최신 공공도서관 원본 확인
library_path = (
    BASE_PATH
    + "01_raw_data/공공서비스_공급/"
    + "부산시 공공도서관 현황(2026.7)(게시용).xlsx"
)

library_excel = pd.ExcelFile(library_path)

print("시트 목록:", library_excel.sheet_names)

for sheet in library_excel.sheet_names:
    temp = pd.read_excel(library_path, sheet_name=sheet, header=None)
    print(f"\n[{sheet}] shape:", temp.shape)
    display(temp.head(10))

In [ ]:
# 실제 표 구조로 공공도서관 데이터 불러오기
library_raw = pd.read_excel(
    library_path,
    sheet_name="Sheet1 (2)",
    header=1
)

print("shape:", library_raw.shape)
print("columns:", library_raw.columns.tolist())

print("\n[결측값]")
print(library_raw.isna().sum())

print("\n[중복]")
print("전체 행 중복:", library_raw.duplicated().sum())
print("도서관명 중복:", library_raw["도서관명"].duplicated().sum())
print("주소 중복:", library_raw["주소"].duplicated().sum())

print("\n[휴관·폐관 등 상태 표기]")
status_mask = library_raw["도서관명"].astype(str).str.contains(
    "휴관|폐관|운영중단",
    regex=True,
    na=False
)
display(library_raw[status_mask])

print("\n[주소에 '부산광역시'가 정상적으로 포함되지 않은 행]")
address_check = ~library_raw["주소"].astype(str).str.contains(
    "부산광역시",
    na=False
)
display(library_raw[address_check])

print("\n[전체 58개 확인]")
display(library_raw)

In [ ]:
# 현재 운영 중인 공공도서관 분석용 데이터 생성
library = library_raw.copy()

library["도서관명"] = library["도서관명"].astype(str).str.strip()
library["주소"] = library["주소"].astype(str).str.strip()

library_active = library[
    ~library["도서관명"].str.contains("휴관|폐관|운영중단", regex=True, na=False)
].copy()

library_active["시설유형"] = "공공도서관"

print("원본 도서관:", len(library))
print("운영 도서관:", len(library_active))
print("제외 도서관:", len(library) - len(library_active))

print("\n분석용 결측:", library_active[["도서관명", "주소"]].isna().sum().sum())
print("도서관명 중복:", library_active["도서관명"].duplicated().sum())
print("주소 중복:", library_active["주소"].duplicated().sum())

display(library_active.head())

In [ ]:
# 분석용 공공도서관 데이터 저장
library_output_path = (
    BASE_PATH
    + "02_processed_data/공공도서관_202607_분석용.csv"
)

library_active.to_csv(
    library_output_path,
    index=False,
    encoding="utf-8-sig"
)

print("저장 완료:", library_output_path)
print("저장 shape:", library_active.shape)

In [ ]:
# 주거·직장·방문 생활인구 구성비 생성
regional_profile["주거인구_구성비"] = (
    regional_profile["평균주거인구수"]
    / regional_profile["전체활동인구"]
)

regional_profile["직장인구_구성비"] = (
    regional_profile["평균직장인구수"]
    / regional_profile["전체활동인구"]
)

regional_profile["방문인구_구성비"] = (
    regional_profile["평균방문인구수"]
    / regional_profile["전체활동인구"]
)

# 구성비 합계 검증
regional_profile["활동구성비_합계"] = (
    regional_profile["주거인구_구성비"]
    + regional_profile["직장인구_구성비"]
    + regional_profile["방문인구_구성비"]
)

print("행정동 수:", len(regional_profile))
print("결측:", regional_profile.isna().sum().sum())
print(
    "구성비 합계 최대 오차:",
    (regional_profile["활동구성비_합계"] - 1).abs().max()
)

display(
    regional_profile[
        [
            "행정동명",
            "전체활동인구",
            "주거인구_구성비",
            "직장인구_구성비",
            "방문인구_구성비"
        ]
    ].head()
)

In [ ]:
# 지역수요 프로필 중간 결과 저장
regional_profile_save = regional_profile.drop(
    columns=["활동구성비_합계"]
).copy()

profile_path = (
    BASE_PATH
    + "02_processed_data/지역수요프로필_행정동별_1차.csv"
)

regional_profile_save.to_csv(
    profile_path,
    index=False,
    encoding="utf-8-sig"
)

print("저장 완료:", profile_path)
print("shape:", regional_profile_save.shape)
print("행정동 수:", regional_profile_save["행정동코드"].nunique())

In [ ]:
# 지역수요 핵심 지표 기술통계
eda_cols = [
    "전체활동인구",
    "20대미만_활동인구",
    "20대미만_구성비",
    "60대이상_활동인구",
    "60대이상_구성비",
    "아동돌봄시설_전체",
    "고령층생활서비스_전체",
    "주거인구_구성비",
    "직장인구_구성비",
    "방문인구_구성비"
]

display(
    regional_profile_save[eda_cols]
    .describe()
    .T
)

In [ ]:
# 연령별 활동인구 규모 상위 지역
print("[20대미만 활동인구 상위 10]")
display(
    regional_profile_save[
        ["행정동명", "20대미만_활동인구", "20대미만_구성비", "아동돌봄시설_전체"]
    ]
    .sort_values("20대미만_활동인구", ascending=False)
    .head(10)
)

print("\n[20대미만 구성비 상위 10]")
display(
    regional_profile_save[
        ["행정동명", "20대미만_활동인구", "20대미만_구성비", "아동돌봄시설_전체"]
    ]
    .sort_values("20대미만_구성비", ascending=False)
    .head(10)
)

print("\n[60대이상 활동인구 상위 10]")
display(
    regional_profile_save[
        ["행정동명", "60대이상_활동인구", "60대이상_구성비", "고령층생활서비스_전체"]
    ]
    .sort_values("60대이상_활동인구", ascending=False)
    .head(10)
)

print("\n[60대이상 구성비 상위 10]")
display(
    regional_profile_save[
        ["행정동명", "60대이상_활동인구", "60대이상_구성비", "고령층생활서비스_전체"]
    ]
    .sort_values("60대이상_구성비", ascending=False)
    .head(10)
)

In [ ]:
# 생활인구 구성 특성 상위 지역
for col, label in [
    ("주거인구_구성비", "주거 중심"),
    ("직장인구_구성비", "직장 중심"),
    ("방문인구_구성비", "방문 중심")
]:
    print(f"\n[{label} 상위 10]")

    display(
        regional_profile_save[
            [
                "행정동명",
                "전체활동인구",
                "주거인구_구성비",
                "직장인구_구성비",
                "방문인구_구성비"
            ]
        ]
        .sort_values(col, ascending=False)
        .head(10)
    )

In [ ]:
# 공공서비스 공급 분포
print("[아동돌봄시설]")
print(regional_profile_save["아동돌봄시설_전체"].describe())
print("공급 0개 행정동:",
      (regional_profile_save["아동돌봄시설_전체"] == 0).sum())

print("\n[고령층생활서비스]")
print(regional_profile_save["고령층생활서비스_전체"].describe())
print("공급 0개 행정동:",
      (regional_profile_save["고령층생활서비스_전체"] == 0).sum())

In [ ]:
# 주요 수요와 공급 간 단순 상관관계 확인
corr_cols = [
    "20대미만_활동인구",
    "20대미만_구성비",
    "아동돌봄시설_전체",
    "60대이상_활동인구",
    "60대이상_구성비",
    "고령층생활서비스_전체"
]

display(
    regional_profile_save[corr_cols]
    .corr()
    .round(3)
)

## 지역수요 프로필 분석 결과

부산 206개 행정동을 대상으로 연령별 생활인구, 주거·직장·방문 생활인구 구성, 아동돌봄 및 고령층 생활서비스 공급 현황을 결합하여 지역별 생활수요 특성을 분석하였다.

### 1. 연령 수요는 절대 규모와 구성비를 함께 볼 필요가 있다
20대 미만 및 60대 이상 생활인구를 비교한 결과, 활동인구 절대 규모가 큰 지역과 해당 연령대의 구성비가 높은 지역이 서로 다르게 나타났다.

따라서 특정 연령층의 수요를 진단할 때 단순한 인구 규모만으로 판단하기보다,
- 실제 잠재 이용자 규모를 나타내는 `활동인구 규모`
- 지역의 인구구조 특성을 나타내는 `연령 구성비`

를 함께 고려할 필요가 있다.

### 2. 행정동별 활동 특성도 뚜렷한 차이를 보였다
주거·직장·방문 생활인구 구성비를 비교한 결과, 행정동별로 주거 중심, 직장 중심, 방문 중심의 활동 특성이 구분되었다.

이는 동일한 인구 규모의 지역이라도 실제 공간이 사용되는 방식은 다를 수 있음을 의미하며, 향후 유휴 공유재산의 활용유형을 판단할 때 지역의 생활·활동 특성을 함께 고려할 필요가 있음을 보여준다.

### 3. 공공서비스 공급은 서비스 유형별 분포 특성이 다르다
아동돌봄시설은 공급이 없는 행정동이 다수 존재한 반면, 고령층 생활서비스는 대부분의 행정동에 분포하였다.

따라서 시설 수가 0이거나 적다는 사실만으로 해당 지역을 곧바로 서비스 부족 지역으로 판단하지 않고, 수요 규모와 구성비, 기존 공급 수준을 함께 해석한다.

### 4. 수요와 공급 간에는 일정한 관계가 있으나 단순 대응하지 않는다
상관관계 분석 결과, 연령별 활동인구 규모와 관련 서비스 공급량 사이에는 일정 수준의 양의 상관관계가 나타났다.

반면 연령 구성비와 공급량 간의 관계는 상대적으로 약하게 나타나, 지역의 연령 특성이 기존 서비스 공급에 그대로 반영되고 있다고 보기는 어렵다.

본 분석에서 상관관계는 서비스 적정성이나 부족 여부를 판정하기 위한 기준이 아니라, 지역수요와 현재 공급 간의 관계를 탐색하기 위한 참고 지표로 활용한다.

## 분석 시사점

본 단계의 목적은 특정 행정동을 단순히 '시설 부족 지역'으로 선정하는 것이 아니라, 각 행정동의 인구구조와 활동 특성, 공공서비스 공급 현황을 하나의 지역수요 프로필로 구축하는 데 있다.

이후 개별 유휴 공유재산의 위치 및 물리적 특성과 해당 지역수요 프로필을 결합하여, 각 재산에 적합한 활용유형과 우선 검토 대상을 도출한다.

In [ ]:
# 최종 지역수요 프로필 저장
final_profile_path = (
    BASE_PATH
    + "02_processed_data/지역수요프로필_행정동별.csv"
)

regional_profile_save.to_csv(
    final_profile_path,
    index=False,
    encoding="utf-8-sig"
)

print("저장 완료:", final_profile_path)
print("shape:", regional_profile_save.shape)